# Demo 1 · Energy: a silent and an active network on one board

Instructor demo. Runs the same network twice on one 48-chip board, once silent (no spikes) and once active (every neuron spikes in every step), and plots the board power during both runs. Start section 2 in the break: the two runs take about 2 minutes each. If the chip is not available, section 3 shows the values measured on 2026-10-19 (`05_measurements/01_power_baseline/`).

The message: the board needs about 57 W just for being on; small networks are invisible in this number; spiking activity is what costs energy.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

N_CORES = 1000  # cores (PEs) of the board to load, out of 7,104
NEURONS_PER_CORE = 100
STEPS = 20000  # 20 s at 1 ms per step
OUTPUT = Path("output")
MEASURED = Path("../../05_measurements/01_power_baseline/data")

## 1. The network

1,000 cores, each holding 100 `lif` neurons connected all-to-all to each other (10,000 synapses per core). **Silent:** no input, nothing spikes, the cores only run their timer loop. **Active:** a constant input above the threshold makes every neuron spike in every step, i.e. 10,000 synaptic events per core and step, 10 billion per second on the board.

In [ ]:
def build(mode):
    from spinnaker2 import snn

    pre, post = np.meshgrid(np.arange(NEURONS_PER_CORE), np.arange(NEURONS_PER_CORE), indexing="ij")
    conns = np.stack([pre.ravel(), post.ravel(), np.ones(pre.size), np.zeros(pre.size)], axis=1).astype(int)
    params = {"threshold": 1.0, "alpha_decay": 0.9, "i_offset": 2.0 if mode == "active" else 0.0}
    net = snn.Network(f"energy_{mode}")
    for k in range(N_CORES):
        pop = snn.Population(size=NEURONS_PER_CORE, neuron_model="lif", params=params, name=f"p{k}", record=[])
        net.add(pop, snn.Projection(pre=pop, post=pop, connections=conns))
    return net


def run(mode):
    """One run with power recording; stores the power trace as output/<mode>.parquet."""
    from spinnaker2 import hardware
    from spinnaker2.tools import diagnostic_folder_utils as dfu

    dfu.set_level("persistent")
    hardware.SpiNNcloud48NodeBoard().run(build(mode), STEPS, record_power=True)
    OUTPUT.mkdir(exist_ok=True)
    shutil.copy(dfu.get_path() / "power.parquet", OUTPUT / f"{mode}.parquet")
    print(f"{mode}: done")

## 2. Run on the chip (start in the break, about 4 minutes)

In [ ]:
for mode in ["silent", "active"]:
    run(mode)

## 3. Board power during the simulation

The power is read from the board's controller about once per second, in steps of 1 W. The first 2 seconds of each run are left out of the averages, because the reading takes up to a second to settle.

In [ ]:
def load(mode):
    """Live run if available, otherwise the measurement of 2026-10-19."""
    live = OUTPUT / f"{mode}.parquet"
    if live.exists():
        return pd.read_parquet(live), "live run"
    name = {"silent": f"silent_{N_CORES}", "active": f"active_{N_CORES}", "empty": "idle_1"}[mode]
    return pd.read_parquet(MEASURED / f"{name}.parquet"), "measured 2026-10-19"


def simulation_window(df):
    marks = dict(zip(df.loc[df.marker.notna(), "marker"], df.loc[df.marker.notna(), "time_s"]))
    sim = df[(df.time_s > marks["sim_start"]) & (df.time_s < marks["sim_end"]) & df.total.notna()]
    return sim.time_s - marks["sim_start"], sim.total


fig, ax = plt.subplots(figsize=(9, 4))
for mode, color in [("empty", "0.5"), ("silent", "#00305C"), ("active", "crimson")]:
    df, source = load(mode)
    t, power = simulation_window(df)
    steady = power[t > 2].mean()
    label = {"empty": "empty board (1 neuron)", "silent": f"{N_CORES} cores, silent", "active": f"{N_CORES} cores, every neuron spikes"}[mode]
    ax.plot(t, power, color=color, label=f"{label}: {steady:.0f} W ({source})")
ax.set(xlabel="time since start of the simulation (s)", ylabel="board power (W)", ylim=(50, 75))
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 4. What it shows

- The empty board needs about 57 W, whatever runs on it. A workshop network of a few cores would read 57 W as well: the 1 W resolution cannot show it.
- Loading 1,000 cores without spikes adds about 3 W: cores that only tick sleep most of the time.
- The same network with every neuron spiking adds a few watts more: spiking activity costs energy, silence is cheap. That is the principle behind neuromorphic computing, and the reason sparse activity matters.
- The exact numbers vary between runs and boards: the active network read 66 W on 2026-10-19 and 63 W in a test run on 2026-11-09 (silent 60 W both times). With 1 W steps and a reading about once per second, single runs differ by a few watts; occasional one-second dips are reading glitches.
- The full measurement with up to 7,000 cores: `05_measurements/01_power_baseline/README.md`.